# Relatório Mensal de Distância, Emissões e Custo de Passagens

Este notebook carrega o arquivo mestre de viagens aéreas, processa as datas e agrupa os totais de distância, emissões e **custo de passagens** por mês, gerando uma tabela e gráficos.

## 1. Importações e Configurações

In [127]:
import pandas as pd
import numpy as np
import altair as alt
import os

# --- Configurações ---
ano = 2025 # Ano dos dados a serem analisados (AJUSTE CONFORME NECESSÁRIO)
# orgao = 'UFPB' # ou 'UFCG'
orgao = 'UFCG'
# Caminho para o arquivo de entrada 
pasta_dados = f'dadosViagens/dados_viagens{ano}'
# Ajusta o nome do arquivo de entrada com base no órgão
arquivo_entrada = os.path.join(pasta_dados, f'df_master_{orgao}_aereo_{ano}.csv') 

# Colunas que serão usadas do arquivo de entrada
COL_ID_VIAGEM = 'Identificador do processo de viagem'
COL_DATA = 'Período - Data de início' 
COL_DISTANCIA = 'Distância (GCD)'
COL_EMISSOES = 'Emissões (KgCO2eq)'
COL_GASTOS_PASSAGENS = 'Valor passagens' # <<< FOCO APENAS EM PASSAGENS >>>


## 2. Carregar e Preparar Dados

In [128]:
print(f"🔄 Carregando dados de: '{arquivo_entrada}'...")
df = pd.DataFrame() # Inicializa df vazio
try:
    df = pd.read_csv(arquivo_entrada)
    print(f"✅ Dados carregados com sucesso ({len(df)} viagens).")
    
    # --- Verificação de Colunas --- 
    colunas_necessarias = [COL_ID_VIAGEM, COL_DATA, COL_DISTANCIA, COL_EMISSOES, COL_GASTOS_PASSAGENS]
    colunas_faltantes = [col for col in colunas_necessarias if col not in df.columns]
    if colunas_faltantes:
        print(f"❌ ERRO: Colunas essenciais não encontradas: {colunas_faltantes}")
        df = pd.DataFrame() # Reseta df se faltar algo crítico
    else:
        print("✅ Colunas necessárias (Data, Distância, Emissões, Passagens) encontradas.")

except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{arquivo_entrada}' não encontrado.")
except Exception as e:
    print(f"❌ Ocorreu um erro ao carregar os dados: {e}")

# --- Processamento de Datas e Custos --- 
if not df.empty:
    print("🔄 Processando datas e custos...")
    # Converte a coluna de data
    df['Data_Viagem'] = pd.to_datetime(df[COL_DATA], format='%d/%m/%Y', errors='coerce')
    
    # --- CORREÇÃO: Converte TODAS as colunas numéricas necessárias ---
    colunas_numericas_processar = [COL_DISTANCIA, COL_EMISSOES, COL_GASTOS_PASSAGENS]
    for col in colunas_numericas_processar:
         # Converte para numérico, tratando vírgulas e erros
         df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(0)
    print("✅ Colunas de Distância, Emissões e Passagens convertidas para numérico.")
    
    # Remove linhas onde a data não pôde ser convertida
    linhas_antes = len(df)
    df.dropna(subset=['Data_Viagem'], inplace=True)
    linhas_removidas = linhas_antes - len(df)
    if linhas_removidas > 0:
        print(f"   - {linhas_removidas} viagens removidas por data inválida.")
        
    # === MUDANÇA APLICADA AQUI ===
    # Cria colunas para agrupar por mês
    # Usa o 'ano' da CÉLULA DE CONFIGURAÇÃO para forçar o ano nos rótulos
    df['Mes_Num'] = df['Data_Viagem'].dt.month # Número 1-12 para ordenação
    df['Mes_Str'] = df['Data_Viagem'].dt.strftime('%m') # Formato '01', '02'
    df['Mes_Ano'] = f"{ano}-" + df['Mes_Str'] # Força o ano, ex: '2025-01'
    
    print("✅ Datas processadas e colunas de Mês criadas (ANO FORÇADO PELO CONFIG).")
    print(df[['Data_Viagem', 'Mes_Ano', 'Mes_Num', COL_GASTOS_PASSAGENS]].head())
    # === FIM DA MUDANÇA ===
else:
    print("⚠️ DataFrame vazio, processamento pulado.")

🔄 Carregando dados de: 'dadosViagens/dados_viagens2025\df_master_UFCG_aereo_2025.csv'...
✅ Dados carregados com sucesso (320 viagens).
✅ Colunas necessárias (Data, Distância, Emissões, Passagens) encontradas.
🔄 Processando datas e custos...
✅ Colunas de Distância, Emissões e Passagens convertidas para numérico.
✅ Datas processadas e colunas de Mês criadas (ANO FORÇADO PELO CONFIG).
  Data_Viagem  Mes_Ano  Mes_Num  Valor passagens
0  2023-02-06  2025-02        2          1745.84
1  2023-02-07  2025-02        2          2453.45
2  2023-01-29  2025-01        1          2936.15
3  2023-03-12  2025-03        3          2274.46
4  2023-02-03  2025-02        2         11452.80


## 3. Agrupar por Mês e Gerar Relatório

In [129]:
df_mensal = pd.DataFrame() # Cria DF vazio por segurança

if not df.empty:
    print(f"🔄 Agrupando totais por Mês/Ano para {ano}...")
    
    # Agrupa por Mês/Ano e soma as colunas de interesse
    df_mensal = df.groupby(['Mes_Ano', 'Mes_Num']).agg(
        Total_Distancia_Km = (COL_DISTANCIA, 'sum'),
        Total_Emissoes_KgCO2eq = (COL_EMISSOES, 'sum'),
        Total_Viagens = (COL_ID_VIAGEM, 'count'),
        Total_Passagens = (COL_GASTOS_PASSAGENS, 'sum') # <<<--- NOME DA COLUNA ATUALIZADO
    ).reset_index()
    
    # Ordena pelo número do mês
    df_mensal.sort_values('Mes_Num', inplace=True)
    
    print("✅ Agrupamento concluído.")
    
    # --- Exibir Tabela de Resultados --- 
    print(f"\n--- Relatório Mensal: Totais de Viagens Aéreas ({orgao} {ano}) ---")
    # <<<--- NOME DA COLUNA ATUALIZADO
    colunas_exibir = ['Mes_Ano', 'Total_Viagens', 'Total_Distancia_Km', 'Total_Emissoes_KgCO2eq', "Total_Passagens"]
    print(df_mensal[colunas_exibir].to_string(index=False, formatters={
        'Total_Distancia_Km': '{:,.0f} km'.format,
        'Total_Emissoes_KgCO2eq': '{:,.0f} kg'.format,
        'Total_Passagens': 'R$ {:,.2f}'.format # <<<--- FORMATADOR CORRETO
    }))
    
    # --- Salvar Relatório CSV --- 
    try:
        nome_arquivo_saida = f"relatorio_mensal_{orgao}_aereo_{ano}.csv"
        caminho_saida = os.path.join(pasta_dados, nome_arquivo_saida)
        df_mensal.round(2).to_csv(caminho_saida, index=False)
        print(f"\n✅ Relatório mensal salvo em: '{caminho_saida}'")
    except Exception as e_save:
        print(f"❌ Erro ao salvar relatório mensal: {e_save}")
        
else:
    print("⚠️ DataFrame vazio, agrupamento pulado.")

🔄 Agrupando totais por Mês/Ano para 2025...
✅ Agrupamento concluído.

--- Relatório Mensal: Totais de Viagens Aéreas (UFCG 2025) ---
Mes_Ano  Total_Viagens Total_Distancia_Km Total_Emissoes_KgCO2eq Total_Passagens
2025-01              1           4,231 km                 847 kg     R$ 2,936.15
2025-02              7          32,717 km               6,252 kg    R$ 31,696.13
2025-03             10          37,852 km               7,220 kg    R$ 37,901.75
2025-04             18          57,196 km              10,787 kg    R$ 62,080.29
2025-05             18          74,251 km              14,133 kg    R$ 50,023.24
2025-06             28         107,720 km              20,436 kg   R$ 113,789.01
2025-07             48         332,246 km              65,517 kg   R$ 189,065.39
2025-08             16          69,624 km              13,526 kg    R$ 50,129.53
2025-09             32         146,929 km              28,893 kg   R$ 122,650.71
2025-10             62         265,922 km              51

## 4. Visualização (Gráficos Mensais)

In [130]:
if 'df_mensal' in locals() and not df_mensal.empty:
    print("🔄 Criando gráficos mensais com rótulos de dados...")
    
    # Gráfico base (para tooltips e ordenação)
    base = alt.Chart(df_mensal).encode(
        # Ordena o eixo X pelo número do mês, mas exibe o nome do mês/ano
        x=alt.X('Mes_Ano', sort=alt.EncodingSortField(field="Mes_Num", op="min", order='ascending'), axis=alt.Axis(title='Mês')), 
        tooltip=['Mes_Ano',
                 alt.Tooltip('Total_Viagens', title='Nº de Viagens'),
                 alt.Tooltip('Total_Distancia_Km', title='Distância (km)', format=',.0f'),
                 alt.Tooltip('Total_Emissoes_KgCO2eq', title='Emissões (kg)', format=',.0f'),
                 alt.Tooltip('Total_Passagens', title='Custo Passagens (R$)', format=',.2f') 
                ]
    )
    
    # --- Gráfico 1: Emissões por Mês ---
    bar_emissoes = base.mark_bar(color='#d9534f').encode(
        y=alt.Y('Total_Emissoes_KgCO2eq', axis=alt.Axis(title='Total Emissões (KgCO2eq)'))
    )
    text_emissoes = base.mark_text(
        dy=15, # Desloca para baixo (dentro da barra)
        color='black', # Cor do texto
        baseline='top'
    ).encode(
        y=alt.Y('Total_Emissoes_KgCO2eq'),
        text=alt.Text('Total_Emissoes_KgCO2eq', format=',.0f') # Formato sem casas decimais
    )
    chart_emissoes = (bar_emissoes + text_emissoes).properties(
        title='Emissões Totais por Mês',
        width=700
    )
    
    # --- Gráfico 2: Distância por Mês ---
    bar_distancia = base.mark_bar(color='#4682b4').encode(
        y=alt.Y('Total_Distancia_Km', axis=alt.Axis(title='Total Distância (Km)'))
    )
    text_distancia = base.mark_text(
        dy=15, 
        color='white', # Branco funciona bem no azul
        baseline='top'
    ).encode(
        y=alt.Y('Total_Datancia_Km'),
        text=alt.Text('Total_Distancia_Km', format=',.0f')
    )
    chart_distancia = (bar_distancia + text_distancia).properties(
        title='Distância Total por Mês',
        width=700
    )

    # --- Gráfico 3: Custo de PASSAGENS por Mês ---
    bar_passagens = base.mark_bar(color='#f0ad4e').encode( # Cor Laranja
        y=alt.Y('Total_Passagens', axis=alt.Axis(title='Total Passagens (R$)'))
    )
    text_passagens = base.mark_text(
        dy=15, 
        color='black', # Preto é melhor no laranja
        baseline='top'
    ).encode(
        y=alt.Y('Total_Passagens'),
        text=alt.Text('Total_Passagens', format=',.0f') # Formato sem casas decimais
    )
    chart_passagens = (bar_passagens + text_passagens).properties(
        title='Custo Total de Passagens por Mês',
        width=700 
    )
    
    # --- Gráfico 4: Número de Viagens por Mês ---
    line_viagens = base.mark_line(color='#5cb85c', point=True).encode(
        y=alt.Y('Total_Viagens', axis=alt.Axis(title='Nº de Viagens'))
    )
    text_viagens = base.mark_text(
        dy=-10, # Desloca para cima (acima do ponto)
        color='black'
    ).encode(
        y=alt.Y('Total_Viagens'),
        text=alt.Text('Total_Viagens', format='d') # Formato inteiro
    )
    chart_viagens = (line_viagens + text_viagens).properties(
        title='Número de Viagens por Mês',
        width=700 
    )

    # Combina os gráficos
    dashboard_mensal = alt.vconcat(
        chart_emissoes, 
        chart_distancia, 
        chart_passagens, 
        chart_viagens
    ).properties(
        title=f"Análise Mensal de Viagens Aéreas - {orgao} {ano}"
    ).resolve_scale(x='shared') # Compartilha o eixo X
    
    # --- Salvar o dashboard como HTML ---
    try:
        pasta_dados_out = f'dadosViagens/dados_viagens{ano}/dashboards_mensais'
        os.makedirs(pasta_dados_out, exist_ok=True)  # ✅ cria a pasta se não existir

        arquivo_dashboard_html = os.path.join(
            pasta_dados_out, f'dashboard_mensal_{orgao}_aereo_{ano}.html'
        )

        dashboard_mensal.save(arquivo_dashboard_html)
        print(f"\n✅ Dashboard mensal salvo como HTML em: '{arquivo_dashboard_html}'")
        print("   (Abra este arquivo .html no seu navegador para ver o gráfico.)")

    except Exception as e_save_chart:
        print(f"❌ Erro ao salvar dashboard mensal como HTML: {e_save_chart}")

else:
    print("⚠️ DataFrame mensal vazio, gráficos pulados.")


🔄 Criando gráficos mensais com rótulos de dados...
❌ Erro ao salvar dashboard mensal como HTML: Unable to determine data type for the field "Total_Datancia_Km"; verify that the field name is not misspelled. If you are referencing a field from a transform, also confirm that the data type is specified correctly.


In [131]:
dashboard_mensal

ValueError: Unable to determine data type for the field "Total_Datancia_Km"; verify that the field name is not misspelled. If you are referencing a field from a transform, also confirm that the data type is specified correctly.

alt.VConcatChart(...)